# Background Theory — Aircraft Valuation & Similar Asset-Pricing Projects

This notebook explains the *why* behind every technique used in the project, with special attention to concepts specific to asset valuation and to a couple of data-cleaning gotchas that are genuinely easy to miss. Small illustrative code cells use synthetic data — they demonstrate the concept, not the actual project dataset.


## 1. How Aircraft Valuation Actually Works

A professional aircraft appraisal (the kind an ISTAT-certified appraiser produces) typically starts from a **base value curve**: for a given aircraft type, what does a brand-new example cost, and how does that value decline with age under "half-life" utilization assumptions? The appraiser then applies a series of **condition adjustments** on top of that baseline: maintenance status (is a major check freshly done or coming due soon?), engine time remaining before overhaul, damage history, and market-demand conditions for that specific type at that specific time.

This structure — a baseline determined by category and age, adjusted multiplicatively by a series of condition factors — is exactly why this project's feature-importance results (Section 12 of the Solutions notebook) show `aircraft_type` and `age_years` dominating, with maintenance-condition features playing a secondary role. It also explains *why* the value-generating process is fundamentally **multiplicative** rather than additive (Section 3 below).


## 2. A Genuine Pandas Gotcha: Default Missing-Value Parsing

`pandas.read_csv` decides what counts as "missing" using a built-in list of strings, which includes (among others) `''`, `'NA'`, `'N/A'`, `'NULL'`, `'NaN'`, `'None'`, `'n/a'`, `'nan'`, and `'null'`. This is convenient when your data really does use one of these as a missing-value marker — but it becomes a trap the moment one of these strings is a **legitimate category value** in your data, as `"None"` is for `damage_history` in this project (meaning "no damage history," not "we don't know").

**Why this matters more than it might seem:** if you don't catch it, you'll silently lose information (a real, informative category becomes indistinguishable from missing data) and your `.isnull().sum()` audit will report a "data quality problem" that isn't real — potentially sending you down the wrong path entirely (e.g. building an unnecessary imputation strategy for a column that didn't actually need one).

**Detection habits that catch this:**
- When any column's `.isnull().sum()` looks suspiciously high relative to what you'd expect from the source, check the *other* (non-null) values in that column and ask whether one of pandas' default NA strings is a plausible legitimate category.
- If a data dictionary or domain knowledge says a column shouldn't have any missing data, but `.isnull()` disagrees, that mismatch is itself worth investigating before writing an imputation strategy.

**Prevention:** if you know in advance a dataset might contain a legitimate `"None"`/`"NA"`/etc. category, load with `pd.read_csv(path, keep_default_na=False, na_values=[...])`, explicitly specifying only the strings that really do mean "missing" for *your* dataset.


In [1]:
import pandas as pd
demo = pd.DataFrame({'status': ['None', 'Minor', 'Major', 'None']})
demo.to_csv('_demo.csv', index=False)

default_load = pd.read_csv('_demo.csv')
print("Default load nulls:", default_load['status'].isnull().sum(), "-- WRONG, these are real categories")

safe_load = pd.read_csv('_demo.csv', keep_default_na=False, na_values=[''])
print("Safe load nulls:", safe_load['status'].isnull().sum(), "-- correct")

import os; os.remove('_demo.csv')


Default load nulls: 2 -- WRONG, these are real categories
Safe load nulls: 0 -- correct


## 3. Multiplicative vs. Additive Processes, and the Log Transform

This is the central modeling lesson of this project, and it's worth contrasting directly with the previous two projects in this series:

- **Laptop pricing** (previous project): price depends on **interactions** between specs (RAM matters more on a premium brand) — tree ensembles won because they naturally represent interactions.
- **Sports-betting spreads** (previous project): the spread is a mostly **additive** combination of team-strength differentials — plain linear regression won because the true relationship really is close to a weighted sum.
- **Aircraft valuation** (this project): value depreciates **multiplicatively** — a percentage decline per year, further adjusted by percentage factors for condition, demand, etc. A plain linear model on the raw dollar target struggles here, but a **log-linear model** (fit on `log(value)`, then exponentiate predictions back to dollars) captures this almost as well as a full Gradient Boosting ensemble, because a multiplicative process becomes additive once you take logs: `log(a × b × c) = log(a) + log(b) + log(c)`.

**How to recognize a multiplicative process before you model it:**
- The target is strictly positive and spans multiple orders of magnitude (aircraft worth $4M sitting alongside aircraft worth $160M).
- The raw target is right-skewed, and `log(target)` is noticeably more symmetric (check both skewness values directly, as in Section 6.1 of the Solutions notebook).
- Domain knowledge says the process is fundamentally about *rates* or *percentages* (depreciation rate, interest compounding, population growth, disease spread) rather than fixed absolute increments.


In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(0)
n = 500
age = rng.uniform(0, 25, n)
condition_factor = rng.uniform(0.85, 1.15, n)
base_value = 50_000_000

# a genuinely multiplicative process: value = base * exp(-age/20) * condition_factor * noise
true_value = base_value * np.exp(-age / 20) * condition_factor * rng.normal(1.0, 0.05, n)

X = pd.DataFrame({'age': age, 'condition_factor': condition_factor})

lin = LinearRegression().fit(X, true_value)
mae_raw = mean_absolute_error(true_value, lin.predict(X))

lin_log = LinearRegression().fit(X, np.log(true_value))
mae_log = mean_absolute_error(true_value, np.exp(lin_log.predict(X)))

print(f"Plain linear regression MAE:  ${mae_raw:,.0f}")
print(f"Log-linear regression MAE:    ${mae_log:,.0f}")


Plain linear regression MAE:  $1,915,714
Log-linear regression MAE:    $1,127,401


On a cleanly multiplicative synthetic process like this, the log-linear model's advantage is dramatic — the plain linear model is forced to approximate a curved (exponential decay) relationship with a straight line, while the log-linear model gets to fit a straight line to what is, after the transform, an actually-straight relationship.

## 4. Ordinal Encoding: A Third Encoding Option Beyond One-Hot and Target Encoding

The previous two projects in this series covered one-hot encoding (low-cardinality, unordered categories) and target/mean encoding (high-cardinality, unordered categories). This project introduces a third case: **ordinal encoding**, for categories with a genuine, domain-known order.

`airframe_check_status`, `damage_history`, and `paint_condition` all have this property — there's an unambiguous "better/worse" direction. Ordinal encoding maps each category to a number that preserves that order (e.g. `Poor=0, Fair=1, Good=2, Excellent=3`), which:

- Lets a **linear model** represent "more of this is better" with a single coefficient, instead of needing separate, unordered coefficients for each one-hot dummy.
- Lets a **tree model** make a single, sensible threshold split (e.g. "paint_score >= 2") instead of needing multiple splits across unordered dummy columns to approximate the same ranking.
- Uses far fewer columns than one-hot encoding would for the same information.

**The risk of ordinal encoding:** if you get the ordering wrong, or if the category *doesn't* actually have a clean single dimension of "better/worse" (e.g. a color, a geographic region), you'll introduce a false, misleading numeric relationship (implying, say, that "green" is numerically "between" "red" and "blue" in some meaningful sense). Always ask *"is there a real, defensible reason to rank these specific categories in this specific order?"* before using ordinal encoding — if the answer isn't a confident yes, use one-hot or target encoding instead.


## 5. The Market-Quote Leakage Audit, Revisited With a New Wrinkle

The sports-betting project introduced the idea that a "market quote" column can be leakage even when it's technically knowable before your target — because using it defeats the purpose of building an *independent* estimate. This project adds a useful refinement: **not all market-quote-like columns carry the same leakage risk, and the reasoning for excluding each one can differ.**

- `recent_comparable_sale_price_usd` (a **settled transaction**) is about as close to "the answer, restated" as a feature can get without literally being the target — exclude it without much debate.
- `broker_asking_price_usd` (a **quote/negotiating position**) is still excluded for this project's specific "build an independent estimate" goal, but the *reasoning* is softer: a broker's ask isn't even meant to be an unbiased estimate of fair value in the first place (it's a starting point for negotiation), which is a different and arguably weaker leakage concern than a settled transaction price would be.

**The general principle to carry forward:** when auditing for market-quote leakage, don't just apply a single correlation threshold and stop there. Ask, for each flagged column, *what kind of information does this represent* (a settled fact vs. an opinion vs. a negotiating position), and let that shape both your inclusion/exclusion decision and how confidently you can justify it in a written report.


## 6. Evaluating Value Predictions Across a Wide Dynamic Range

When a target spans multiple orders of magnitude (regional jets to widebodies, in this project), a single MAE number can be misleading: a model might have excellent proportional accuracy on cheap items and terrible proportional accuracy on expensive ones (or vice versa) while still producing a "reasonable-looking" average MAE dominated by the largest-value examples.

**MAPE becomes a genuinely useful headline metric in exactly this situation** — unlike the sports-betting project, where MAPE was unstable because the target could sit near zero, here the target is always comfortably positive (aircraft are never worth ~$0), so a percentage-based error metric behaves well and gives a fairer sense of relative accuracy across categories.

**A residual-vs-predicted plot showing a "funnel" (growing absolute error as predicted value increases) is not automatically alarming** in a wide-dynamic-range setting — check whether the *percentage* error also grows before concluding there's a real problem. If percentage error is roughly constant across the range while absolute error grows, that's actually the expected, healthy pattern for a well-behaved multiplicative-process model.


## 7. Backtesting a 'Value-Finding' Decision Rule

This project's backtest (comparing the model's disagreement with a broker's asking price against a held-out comparable-sale price) follows the same methodology as the sports-betting project's, with one instructive difference: **the confirmation signal here (a comparable sale price) is itself noisy and imperfect**, not a clean win/loss outcome. A comparable sale being above the asking price doesn't *prove* the original aircraft was underpriced — it's evidence, weighted by how comparable the "comparable" really was.

This is a common feature of real asset-valuation and trading backtests: you often don't have a clean, unambiguous "did this specific decision pay off" outcome the way a sports bet resolves definitively. Instead, you validate against **proxy evidence** (a related transaction, a subsequent appraisal, a later resale) that's informative but not conclusive. When reporting a backtest like this, it's worth being explicit that the "confirmation rate" measures agreement with a noisy proxy, not a certified ground truth — which is exactly why the write-up in this project's Solutions notebook (Section 13) is careful to frame the result as "encouraging evidence," not "proof of an edge."


## 8. How This Generalizes Beyond Aircraft

The same shape — audit (including checking for parsing gotchas like Section 2's) → clean/engineer with appropriate encoding for each category type → check for a multiplicative process and consider a log transform → audit for market-quote leakage with nuanced, column-by-column reasoning → compare model families including a transformed-target linear model → evaluate with metrics matched to the target's dynamic range → backtest against proxy evidence where a clean outcome isn't available → persist for reuse — applies to:

- **Other capital-asset valuation** (ships, heavy equipment, real estate, data-center hardware): depreciation-driven, multiplicative value processes and "appraisal vs. broker/listing price" leakage structure repeat almost unchanged.
- **Any pricing project with wide dynamic range** (enterprise software licensing, industrial equipment, art/collectibles): the log-transform and MAPE-over-MAE lessons apply directly.
- **Predictive maintenance adjacent work** (which this project touches via engine-hours-since-overhaul and check-status features): the same ordinal-encoding treatment applies to any maintenance-condition category with a natural "how close to due" ordering.
- **Any domain with a genuine `pd.read_csv` string-collision risk** (medical data with a "None" allergy category, survey data with a "N/A — prefer not to answer" option that's meaningfully different from a truly blank response): the Section 2 detection habit — investigate suspiciously-high null counts before trusting them — applies universally, well beyond aerospace.

What changes across domains: the specific base-value curve and condition-adjustment factors, and which columns represent settled facts vs. quotes vs. proxies. What stays constant: matching your target transform to the true generating process, choosing the encoding scheme that matches each category's actual structure (ordered vs. unordered, low vs. high cardinality), and reasoning carefully — column by column — about what "leakage" really means for your project's specific goal.
